# Super Nodes

A **super node** is a published workflow embedded inside another workflow as a single, reusable
node. When the parent reaches it, the runtime **expands** the super node into its whole
sub-workflow, runs it to completion, then continues the parent.

This notebook walks the full composition story end to end:

1. Build a small **2-node sub-workflow** (`create_from_config`)
2. **Publish it as a super node** (`client.super_nodes.publish`)
3. Build a **parent workflow** with one regular node **plus** that super node
4. Wire a **conditional edge** from the regular node into the super node
5. **Run** the parent and watch control hand off into the super node
6. **Clean up** every resource

> The typed config classes come from `interactly.configs` (requires `pip install "interactly[configs]"`).

> **These notebooks are async-first.** They use `AsyncWorkflowClient` with top-level `await`,
> which runs directly in Jupyter (the setup cell calls `nest_asyncio.apply()`). Every method
> shown also exists on the synchronous `WorkflowClient` — just drop the `await`. See the
> [docs](../docs/README.md) for the sync surface. Notebook bodies stay 100% async — there is
> no per-notebook sync cell.

In [ ]:
import _bootstrap  # noqa: F401 - enables import interactly (no install needed)

import os, sys
import nest_asyncio
from dotenv import load_dotenv
from pathlib import Path

# This is required to run asyncio in Jupyter Notebook
nest_asyncio.apply()

# Get the current notebook directory and find the project root
current_dir = Path(os.getcwd())
project_root = current_dir
while project_root.parent != project_root:
    if (project_root / '.env').exists():
        break
    project_root = project_root.parent
else:
    project_root = current_dir
    for _ in range(5):
        if (project_root / 'pyproject.toml').exists():
            break
        project_root = project_root.parent

# Load the environment variables from .env in project root
env_path = project_root / '.env'
if env_path.exists():
    load_dotenv(dotenv_path=str(env_path), override=True)
    print(f"Loaded .env from: {env_path}")
else:
    print(f"Warning: .env file not found at {env_path}")

# Add the project root to sys.path so imports resolve
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
    print(f"Added to Python path: {project_root}")
else:
    print(f"Project root already in Python path: {project_root}")

In [ ]:
# Interactly credentials are read from environment variables.
#
# Convenience defaults point at the dev "Workflow Illustrations" org; your shell
# environment always wins (setdefault only fills in what you have not set).
os.environ.setdefault("INTERACTLY_BASE_URL", "https://api-dev.interactly.ai/workflows")
os.environ.setdefault("INTERACTLY_TEAM_ID", "67458e762b7d3dc15aaea5b5")
os.environ.setdefault("INTERACTLY_USER_ID", "687b1a4f745c8e6806c98d91")

# The bearer token is a secret — never hardcode it in the notebook.
# Export it before launching Jupyter:  export INTERACTLY_API_KEY="…"
assert os.environ.get("INTERACTLY_API_KEY"), (
    "Set INTERACTLY_API_KEY in your environment before running this notebook."
)

#print(f"API KEY is: {os.getenv('INTERACTLY_API_KEY')}")
print(f"TEAM ID is: {os.getenv('INTERACTLY_TEAM_ID')}")
print(f"USER ID is: {os.getenv('INTERACTLY_USER_ID')}")
print(f"BASE URL is: {os.getenv('INTERACTLY_BASE_URL')}")

In [ ]:
import _bootstrap  # noqa: F401 - enables import interactly (no install needed)

import json

import nest_asyncio

from interactly import AsyncWorkflowClient

# Required to run top-level `await` inside a Jupyter notebook
nest_asyncio.apply()

client = AsyncWorkflowClient()
print("Connected to", client._base_url)

## 1. Build a 2-node sub-workflow

Any workflow can become a super node. We build a tiny **appointment-booking** flow: an LLM node
that asks for a preferred date, a `DirectEdgeConfig` to a second LLM node that confirms the
booking. Creating it publishes an active `v0` we can publish as a super node next.

In [ ]:
from interactly.configs import (
    SayLLMNodeConfig,
    PromptConfig,
    DirectEdgeConfig,
    ConditionalEdgeConfig,
    ConditionConfig,
    SuperNodeConfig,
    WorkflowConfig,
    WorkflowConfigFullyHydrated,
    OpenAILLMConfig,
    OPENAIModel,
)
from interactly.types.workflows.workflow import Workflow

booking_llm = OpenAILLMConfig(model=OPENAIModel.GPT_5_4_MINI, max_tokens=300)

ask_date_node = SayLLMNodeConfig(
    name="Ask Preferred Date",
    description="Asks the caller which day and time they would like to come in.",
    is_start=True,
    self_loop=False,
    wait_for_user_message=True,
    main_response_config=PromptConfig(prompt=(
        "You are booking a clinic appointment. Ask the caller which day and time they would "
        "like to come in. Ask exactly one short question."
    )),
    llms_config=booking_llm,
)

confirm_node = SayLLMNodeConfig(
    name="Confirm Booking",
    description="Reads the requested date back and confirms the appointment.",
    self_loop=False,
    wait_for_user_message=True,
    main_response_config=PromptConfig(prompt=(
        "The caller has given a preferred date and time. Read it back to them, confirm the "
        "appointment is booked, and say they will get a reminder. Keep it under 30 words."
    )),
    llms_config=booking_llm,
)

edge_ask_to_confirm = DirectEdgeConfig(
    name="Ask Date -> Confirm",
    source_node_logical_id=ask_date_node.logical_id,
    destination_node_logical_id=confirm_node.logical_id,
)

sub_config = WorkflowConfigFullyHydrated(
    workflow_config=WorkflowConfig(
        name="Appointment Booking (08_super_nodes)",
        description="Reusable 2-node booking sub-workflow, published as a super node.",
    ),
    node_configs=[ask_date_node, confirm_node],
    edge_configs=[edge_ask_to_confirm],
)

sub_wf: Workflow = await client.workflows.create_from_config(sub_config)
SUB_WF_ID = sub_wf.id
print(f"Sub-workflow created  id={SUB_WF_ID}  name={sub_wf.name!r}")

## 2. Publish it as a super node

`publish()` records a `SuperNodeInterface` (the inputs a parent may supply, each `map_to` a
sub-workflow variable) **plus a hydrated snapshot** of the sub-workflow's config. That snapshot is
what a parent embeds and what the runtime expands at execution time. Here the interface declares no
inputs — the parent leaves `field_values` empty and lets the sub-workflow collect the date
conversationally.

In [ ]:
# No input fields for this demo; a real interface would declare fields that map_to
# sub-workflow variables, e.g. {"name": "patient_name", "map_to": "dynamic_variable:patient_name"}.
interface = {"field_values": []}

result = await client.super_nodes.publish(SUB_WF_ID, interface=interface)
VERSION = result["workflow_version_number"]
print(f"Published as a super node — version {VERSION}, {result.get('num_input_fields')} input field(s)")

## 3. Build a parent workflow: a regular node + the super node

The parent is a clinic front desk. It has **one regular node** — a `SayLLMNodeConfig` greeter that
welcomes the caller and self-loops while chatting — and **one super node** that references the
published booking workflow by `super_workflow_id`.

We fetch the published super node with `get()` to copy its recorded **interface** onto the
`SuperNodeConfig` (the runtime needs it to resolve `field_values`). We do **not** embed the config
snapshot: because both workflows live in the same team, the runtime resolves the published
sub-workflow from its id at expansion time. `field_values` is left empty, so the booking
sub-workflow asks for the date itself when control reaches it.

In [ ]:
# Fetch the published super node to copy its interface onto the embedded reference.
published = await client.super_nodes.get(SUB_WF_ID)

GREETER_PROMPT = (
    "You are the front desk greeter at a clinic.\n"
    "- Warmly greet the caller and ask how you can help.\n"
    "- If the caller wants to book, schedule, or set up an appointment, hand off to the booking flow.\n"
    "- Otherwise answer briefly and keep the conversation going."
)

greeter_node = SayLLMNodeConfig(
    name="Front Desk Greeter",
    description="Greets the caller and routes appointment requests into the booking super node.",
    is_start=True,
    self_loop=True,
    wait_for_user_message=True,
    main_response_config=PromptConfig(prompt=GREETER_PROMPT),
    llms_config=OpenAILLMConfig(model=OPENAIModel.GPT_5_4_MINI, max_tokens=300),
)

booking_super_node = SuperNodeConfig(
    name="Appointment Booking (Super Node)",
    description="The published booking sub-workflow, embedded as a single node.",
    # Reference the published super node by id; the runtime resolves it at expansion time.
    super_workflow_id=str(SUB_WF_ID),
    super_node_interface=published.super_node_interface,
    # field_values left empty: the sub-workflow collects the date conversationally.
    field_values={},
)
print(f"Regular node : {greeter_node.name!r}")
print(f"Super node   : {booking_super_node.name!r} -> sub-workflow id={booking_super_node.super_workflow_id}")

## 4. Connect the regular node to the super node with a conditional edge

A `ConditionalEdgeConfig` leaves the greeter and enters the super node **only** when its
natural-language `condition` matches — here, when the caller wants to book an appointment. Any
other message keeps the greeter looping. Creating the parent publishes its active version.

In [ ]:
enter_booking_edge = ConditionalEdgeConfig(
    name="Greeter -> Booking (Super Node)",
    source_node_logical_id=greeter_node.logical_id,
    destination_node_logical_id=booking_super_node.logical_id,
    condition=ConditionConfig(
        condition_freeform="The user wants to book or schedule an appointment."
    ),
)

parent_config = WorkflowConfigFullyHydrated(
    workflow_config=WorkflowConfig(
        name="Clinic Front Desk (08_super_nodes)",
        description="Regular greeter node with a conditional edge into a booking super node.",
    ),
    node_configs=[greeter_node, booking_super_node],
    edge_configs=[enter_booking_edge],
)

parent_wf: Workflow = await client.workflows.create_from_config(parent_config)
PARENT_WF_ID = parent_wf.id
print(f"Parent workflow created  id={PARENT_WF_ID}  name={parent_wf.name!r}")

## 5. Run the parent and watch the hand-off

We drive turns with an `AsyncWorkflowHandle`. The greeter answers first; when the caller asks to
book, the conditional edge fires and control moves **into the super node** — the next question
(the preferred date) comes from the booking sub-workflow's `Ask Preferred Date` node, not the
greeter. A final turn lands on `Confirm Booking`.

> Conditional edges route via an internal tool-call mechanism (names starting with
> `Destination_Node...`); the helper filters those out.
>
> **Note:** executing a workflow that embeds a super node requires an Interactly server that keeps
> the super node's `super_workflow_id` when the referenced workflow lives in the same team, so the
> runtime can resolve and inline the published sub-workflow. On an older server the turns below
> raise a super-node error; sections 1–4 still work.

In [ ]:
from langchain_core.messages import HumanMessage

from interactly.runtime.handle import AsyncWorkflowHandle
from interactly.runtime.events import (
    AssistantResponseEvent,
    BusyWaitForUserMessageEvent,
)
from interactly.configs import (
    LLMNodeRunInput,
    NodesRunInputs,
    WorkflowCommand,
    WorkflowRunInput,
)

chat: AsyncWorkflowHandle = await client.workflows.handle(PARENT_WF_ID)


async def send_message(user_text: str, *, command: WorkflowCommand = WorkflowCommand.DATA):
    """Send one user turn, print the assistant replies, and return the raw events."""
    print(f"User: {user_text}")

    run_input = WorkflowRunInput(
        command=command,
        thread_to_node_inputs={
            "0": NodesRunInputs(
                node_run_inputs=[LLMNodeRunInput(messages=[HumanMessage(content=user_text)])]
            )
        },
    )

    events = []
    async for event in chat.arun(run_input):
        events.append(event)
        if isinstance(event, AssistantResponseEvent) and event.content:
            print(f"  Assistant: {event.content}")
        elif isinstance(event, BusyWaitForUserMessageEvent):
            print("  (waiting for the next user message)")
    return events

### Turn 1 — greeting

The first turn uses `WorkflowCommand.START`. The greeter (the regular node) responds.

In [ ]:
turn1_events = await send_message("Hi there!", command=WorkflowCommand.START)

print(f"\n\n----Events from Turn 1----")
for event in turn1_events:
    print(f"type={type(event).__name__}: {event.model_dump_json(indent=2)}")

### Turn 2 — cross the conditional edge into the super node

Asking to book satisfies the edge condition. Control enters the super node, and the reply is the
date question from the booking sub-workflow — the hand-off in action.

In [ ]:
turn2_events = await send_message("Yes, I'd like to book an appointment.")


print(f"\n\n----Events from Turn 2----")
for event in turn2_events:
    print(f"type={type(event).__name__}: {event.model_dump_json(indent=2)}")

### Turn 3 — finish inside the super node

Providing a date moves the sub-workflow from `Ask Preferred Date` to `Confirm Booking`.

In [ ]:
turn3_events = await send_message("Next Tuesday morning works for me.")


print(f"\n\n----Events from Turn 3----")
for event in turn3_events:
    print(f"type={type(event).__name__}: {event.model_dump_json(indent=2)}")

    
print(f"\nSession run_id: {chat.run_id}")

## 6. Cleanup

Delete the parent (which references the super node), unpublish the super-node interface, then
delete the sub-workflow, and close the client.

In [ ]:
await client.workflows.delete(PARENT_WF_ID)
print(f"Parent workflow {PARENT_WF_ID} deleted.")

unpub = await client.super_nodes.unpublish(SUB_WF_ID, version_number=VERSION)
print(f"Super node unpublished (still referenced by: {unpub.get('workflows_referencing')}).")

await client.workflows.delete(SUB_WF_ID)
print(f"Sub-workflow {SUB_WF_ID} deleted.")

await client.close()

## See also

- Example: [`../wf_examples/wf_example_progression_17.py`](../wf_examples/wf_example_progression_17.py) — embedding a super node inline with `encapsulated_workflow_config`
- [`05_workflow_with_tools.ipynb`](05_workflow_with_tools.ipynb) — the `create_from_config` + chat-handle pattern
- Guide: [`../docs/guides/super_nodes.md`](../docs/guides/super_nodes.md) — the full `client.super_nodes` API